# Study note: HyDE with an in-memory vector store

## What is this example doing?

This notebook builds a small question-answering workflow. It uses **HyDE** (Hypothetical Document Embeddings) to find a useful source document, then asks a language model to answer from that real source.

## Why use HyDE?

A user's question may be short or use different words from the documents. For example, "stop a rental early" may not share wording with a formal source about "premature cancellation of leasehold estates." HyDE first creates a longer, formal-sounding hypothetical passage. Its wording can make a better search query for the vector store.

> The hypothetical passage is only a search aid. It may contain guesses, so it must not be treated as evidence.

## How does information flow?

1. **Prepare** a shared state, text embeddings, example source documents, and a chat model.
2. **Generate** a hypothetical passage from the user's question.
3. **Retrieve** the most similar real source document using that passage.
4. **Answer** the original question using the retrieved source as context.
5. **Display** the hypothetical passage, retrieved source, and final answer so you can compare them.

The main idea is: **use generated text to improve the search, but use retrieved source text to support the answer.**

In [1]:
# Load API keys and other settings from a local .env file.
from dotenv import load_dotenv
load_dotenv()

# Types and LangChain/LangGraph building blocks used below.
from typing import List
from typing_extensions import TypedDict
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv, find_dotenv

# Also look for .env starting from the current working directory.
load_dotenv(find_dotenv(usecwd=True))

True

## 1. Prepare the data and model

**What:** The next cell defines the fields passed through the workflow, creates an embedding-backed in-memory search index, adds two example documents, and initializes the chat model.

**Why:** The graph needs a predictable place to store each step's inputs and results. Embeddings represent text as numbers so the vector store can compare meaning rather than requiring exact keyword matches.

**How to read the state:** `query` is the user's question; `hypothetical_doc` is generated for search; `documents` holds real retrieved sources; `final_answer` is the response. The in-memory index is temporary: it is rebuilt when this notebook runs again.

In [2]:
# 1. SET UP THE DATA PASSED BETWEEN GRAPH STEPS
# TypedDict documents the keys and value types each node reads or returns.
class HyDEState(TypedDict):
    query: str               # The raw user question
    hypothetical_doc: str    # The fake LLM answer used for embedding
    documents: List[Document]# The real retrieved documents
    final_answer: str        # The final grounded output

# Create embeddings (numeric representations of text) and an in-memory
# search index. This small example's source documents disappear on restart.
embeddings = OpenAIEmbeddings()
vectorstore = InMemoryVectorStore(embeddings)
# These are the only source documents the retriever can search in this demo.
vectorstore.add_documents([
    Document(page_content="Procedures governing the premature cancellation of leasehold estates require a written 30-day notice."),
    Document(page_content="Standard office infrastructure software requests must clear engineering approval.")
])

# The same chat model writes the hypothetical text and the final answer.
# Temperature controls variation; a lower value is generally more repeatable.
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

## 2. Define the workflow steps

A **node** is a function that reads values from the shared state and returns new values. LangGraph merges those returned fields into the state for the next node.

- `generate_hypothetical_node`: turns the short question into a formal hypothetical passage. This is useful because its wording may resemble the source text more closely.
- `retrieve_node`: searches the vector store with that passage and returns the closest real document (`k=1` means one result).
- `generate_answer_node`: gives the original question and retrieved document to the model. The answer prompt says to use only that context, keeping the generated hypothetical passage out of the evidence.

**Follow the data:** `query` → `hypothetical_doc` → `documents` → `final_answer`.

In [3]:
# ==========================================
# 2. DEFINE THE THREE STEPS (NODES) IN THE WORKFLOW
# ==========================================

def generate_hypothetical_node(state: HyDEState):
    """Step 1: Write a fake answer matching the text style of the source DB."""
    # Read the original question from the shared workflow state.
    query = state["query"]
    
    hyde_prompt = ChatPromptTemplate.from_template(
        "Write a concise, formal paragraph answering the following question. "
        "Do not include introductory commentary, match the style of a legal or technical document.\n\n"
        "Question: {query}"
    )
    
    # Connect the prompt to the model, then ask it to draft a formal-sounding
    # hypothetical answer. HyDE uses this richer text to improve retrieval;
    # this generated paragraph is not itself treated as a verified source.
    chain = hyde_prompt | llm
    response = chain.invoke({"query": query})
    # Return only this node's new state field; LangGraph keeps it for later nodes.
    return {"hypothetical_doc": response.content}


def retrieve_node(state: HyDEState):
    """Step 2: Embed the fake answer and search the real database."""
    # Search with the hypothetical text, not the user's short raw question.
    fake_text = state["hypothetical_doc"]
    
    # Find the single closest source document by embedding similarity (k=1).
    retrieved_docs = vectorstore.similarity_search(fake_text, k=1)
    # Keep the actual source document in state for the final-answer step.
    return {"documents": retrieved_docs}


def generate_answer_node(state: HyDEState):
    """Step 3: Answer the user's actual question using the discovered real text."""
    # Keep the user's actual question for the response, and use retrieved
    # documents as evidence rather than using the hypothetical paragraph.
    query = state["query"]
    real_docs = state["documents"]
    
    # Combine retrieved document text into the context supplied to the model.
    context = "\n\n".join([doc.page_content for doc in real_docs])
    
    answer_prompt = ChatPromptTemplate.from_template(
        "You are a helpful assistant. Answer the user's question using ONLY the provided context.\n\n"
        "Context:\n{context}\n\n"
        "Question: {query}"
    )
    
    # Ask for an answer grounded only in the retrieved source text.
    chain = answer_prompt | llm
    response = chain.invoke({"query": query, "context": context})
    # Return the final answer as another field in the shared graph state.
    return {"final_answer": response.content}

## 3. Connect and compile the graph

**What:** The next cell registers the three node functions and connects them in order, from `START` to `END`.

**Why:** Defining the functions describes each individual job; the graph describes how those jobs work together. The edges make the sequence explicit, and compiling creates the runnable `app`.

**How:** Read each `add_edge(from, to)` as "run `to` after `from`." This example is a straight line with no branching.

In [4]:
# ==========================================
# 3. CONNECT THE STEPS AND COMPILE THE WORKFLOW
# ==========================================
workflow = StateGraph(HyDEState)

# Register each Python function under a readable node name.
workflow.add_node("hyde_generator", generate_hypothetical_node)
workflow.add_node("retriever", retrieve_node)
workflow.add_node("answer_generator", generate_answer_node)

# Set the order: draft hypothetical text -> retrieve real source -> answer.
workflow.add_edge(START, "hyde_generator")
workflow.add_edge("hyde_generator", "retriever")
workflow.add_edge("retriever", "answer_generator")
workflow.add_edge("answer_generator", END)

# Compile the graph into an app that can be invoked with an initial state.
app = workflow.compile()

## 4. Run the example and inspect the results

**What:** `app.invoke` starts the graph with the user's question. The returned `result` contains the original input plus fields produced by the nodes.

**Why print all three values?** Comparing them makes the HyDE idea visible: the hypothetical passage is a search query, the vector store returns an actual source, and the final answer is generated from that source.

**How to interpret this demo:** The hypothetical paragraph may say more than the source does. The actual source here only supports the written 30-day notice detail. In a real application, retrieval can still return an irrelevant document, so check the retrieved source and do not assume that a fluent model answer is automatically correct.

In [5]:
# ==========================================
# 4. RUN THE SYSTEM
# ==========================================
# This short question does not use the formal wording in the source document.
user_question = "How do I stop a rental early?"

# Run the entire graph. The returned dictionary contains the initial input
# plus values added by each node: hypothetical_doc, documents, final_answer.
result = app.invoke({"query": user_question})

# Show the generated retrieval aid. It can contain guesses, so it is not the
# authoritative answer; the source document below is the retrieved evidence.
print("--- HYPOTHETICAL PARAGRAPH GENERATED BY LLM ---")
print(result["hypothetical_doc"])

print("\n--- REAL SOURCE DOCUMENT FOUND IN VECTORSTORE ---")
# Print the first (and in this demo, only requested) matching source document.
print(result["documents"][0].page_content)

# This is the response generated from the original question and retrieved text.
print("\n--- FINAL SYSTEM ANSWER ---")
print(result["final_answer"])

--- HYPOTHETICAL PARAGRAPH GENERATED BY LLM ---
To terminate a rental agreement prior to its scheduled expiration, the tenant must provide written notice to the landlord in accordance with the terms specified in the lease, including any required notice period. It is essential to review the lease for specific clauses regarding early termination, which may outline conditions such as penalties or the necessity for the landlord to mitigate damages by re-renting the premises. If the lease includes an early termination clause, compliance with its stipulations is mandatory. Additionally, it is advisable to document all communications and agreements made between the tenant and landlord to ensure a clear record of the termination process.

--- REAL SOURCE DOCUMENT FOUND IN VECTORSTORE ---
Procedures governing the premature cancellation of leasehold estates require a written 30-day notice.

--- FINAL SYSTEM ANSWER ---
To stop a rental early, you need to provide a written 30-day notice.
